# 📄 Document RAG Pipeline — Google Colab

An end-to-end **Retrieval-Augmented Generation (RAG)** system.

**Live Demo:** https://document-app-rag-7zrsnhmephfzjq7zbvywwv.streamlit.app/

This notebook builds, step by step:

1. **Document ingestion** — PDFs, raw `.txt` files, or a Hugging Face dataset (domain-specific archive)
2. **Chunking** — clean recursive-character chunking of unstructured text
3. **Embedding** — mapping chunks to dense vector representations
4. **Vector store** — a FAISS index for fast similarity search
5. **Query embedding route** — turning a user question into a query vector
6. **Retrieval module** — pulling the most relevant chunks for a query
7. **Grounded generation** — combining retrieved context + query into an LLM prompt
8. **Optimization experiments** — chunk-size tuning, hybrid (keyword + vector) search, and re-ranking


## ⚙️ Setup — install dependencies

In [3]:
!pip install -q -U langchain langchain-community langchain-huggingface langchain-text-splitters \
    pypdf sentence-transformers faiss-cpu transformers datasets accelerate rank_bm25 requests==2.32.4 langgraph langgraph-prebuilt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.1 MB/s eta 0:00:00


In [31]:
import os
import tempfile
from pathlib import Path

IN_COLAB = "google.colab" in str(get_ipython())
print("Running in Colab:", IN_COLAB)


Running in Colab: True


## 1️. Document Ingestion Module

Accepts three kinds of custom inputs:

- **PDFs** (`.pdf`) via `PyPDFLoader`
- **Raw text files** (`.txt`) via `TextLoader`
- **Domain-specific Hugging Face archives** (any dataset on the Hugging Face Hub) via `datasets.load_dataset`,
  wrapped into LangChain `Document` objects so the rest of the pipeline treats them identically.

All loaders return a `List[Document]`, each with `.page_content` and `.metadata`.


In [32]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_core.documents import Document
from datasets import load_dataset


def load_document(file_path: str):
    """Load a single PDF or TXT file and return a list of LangChain Document objects."""
    file_extension = Path(file_path).suffix.lower()

    if file_extension == ".pdf":
        loader = PyPDFLoader(file_path)
    elif file_extension == ".txt":
        loader = TextLoader(file_path, encoding="utf-8")
    else:
        raise ValueError(f"Unsupported file type: {file_extension}")

    return loader.load()


def load_from_huggingface(dataset_name: str, text_column: str, split: str = "train",
                           max_docs: int = 200, config_name: str = None):
    """
    Load a domain-specific Hugging Face dataset and convert it into LangChain Document
    objects, so it can be chunked/embedded/indexed exactly like a PDF or TXT file.

    Example: load_from_huggingface("squad", text_column="context", max_docs=100)
    """
    ds = load_dataset(dataset_name, config_name, split=split) if config_name \
        else load_dataset(dataset_name, split=split)

    ds = ds.select(range(min(max_docs, len(ds))))

    documents = []
    for i, row in enumerate(ds):
        text = row[text_column]
        if not text:
            continue
        documents.append(
            Document(
                page_content=str(text),
                metadata={"source": dataset_name, "row_id": i},
            )
        )
    return documents


def ingest_documents(sources):
    """
    Generic ingestion dispatcher.

    `sources` is a list where each item is either:
      - a file path string (".pdf" or ".txt"), or
      - a dict describing a Hugging Face dataset:
        {"type": "huggingface", "dataset_name": ..., "text_column": ..., "split": "train", "max_docs": 200}

    Returns the combined list of Document objects from every source.
    """
    all_documents = []
    for source in sources:
        if isinstance(source, dict) and source.get("type") == "huggingface":
            docs = load_from_huggingface(
                dataset_name=source["dataset_name"],
                text_column=source["text_column"],
                split=source.get("split", "train"),
                max_docs=source.get("max_docs", 200),
                config_name=source.get("config_name"),
            )
        else:
            docs = load_document(source)
        all_documents.extend(docs)
    return all_documents

print("Ingestion module ready: load_document(), load_from_huggingface(), ingest_documents()")


Ingestion module ready: load_document(), load_from_huggingface(), ingest_documents()


### Upload your own document

Run the cell below to upload a PDF or TXT file from your machine (Colab file picker).


In [33]:
uploaded_paths = []

if IN_COLAB:
    from google.colab import files
    print("Upload a PDF or TXT file.")
    try:
        uploaded = files.upload()
        for fname in uploaded.keys():
            uploaded_paths.append(fname)
    except Exception as e:
        print("Error uploading file:", e)

print("Files to ingest:", uploaded_paths)

Upload a PDF or TXT file.


Saving book review.pdf to book review (1).pdf
Files to ingest: ['book review (1).pdf']


In [34]:
import logging
# Suppress pypdf logs and warnings
logging.getLogger("pypdf").setLevel(logging.ERROR)
sources = uploaded_paths
documents = ingest_documents(sources)

## **2️.** Chunking

Breaks the unstructured raw text into smaller, overlapping chunks using
RecursiveCharacterTextSplitter.

In [35]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def chunk_documents(documents, chunk_size: int = 1000, chunk_overlap: int = 150):
    """Split a list of LangChain Document objects into smaller, overlapping chunks."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    return text_splitter.split_documents(documents)


chunks = chunk_documents(documents, chunk_size=1000, chunk_overlap=150)

print(f"Total Chunks Created: {len(chunks)}")
print("\nFirst Chunk Preview:\n", chunks[0].page_content[:500])


Total Chunks Created: 10

First Chunk Preview:
 A Book Review on MURDER ON THE ORIENT EXPRESS By  Agatha Christie Submitted by ALMAREDDY SRI RAM REDDY 23B81A05GZ  
   Supervised by Dr. Ch. Srinivas, Sr. Assistant Professor Dr. Rajisha Menon, Associate Professor Dr. T. Kalpana Bharathi, Sr. Assistant Professor  Advanced English Communication and Soft Skills Lab   DEPARTMENT OF CSE CVR COLLEGE OF ENGINEERING (An Autonomous institution, NAAC Accredited and Affiliated to JNTUH, Hyderabad) Vastunagar, Mangalpalli (V), Ibrahimpatnam (M), Rangareddy


## 3️. Embedding Module

Maps each chunk of text into a dense vector using a pre-trained sentence-transformer
model (`all-MiniLM-L6-v2` — small, fast, and good general-purpose quality).


In [36]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import logging
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

from langchain_huggingface import HuggingFaceEmbeddings


def get_embedding_model(model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
    return HuggingFaceEmbeddings(model_name=model_name)


embedding_model = get_embedding_model()

# embed a single chunk and look at the vector shape
sample_vector = embedding_model.embed_query(chunks[0].page_content)
print("Embedding dimension:", len(sample_vector))
print("First 8 values:", sample_vector[:8])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimension: 384
First 8 values: [-0.029453495517373085, 0.04893812909722328, -0.0011315783485770226, 0.009555171243846416, 0.049829285591840744, 0.026589060202240944, -0.011987807229161263, -0.011246850714087486]


## 4️. Vector Database Initialization

Stores every chunk's embedding in a **FAISS** index configured for fast approximate/exact
similarity search, and offers save/load helpers so the index doesn't need to be rebuilt
every run.


In [37]:
from langchain_community.vectorstores import FAISS


def build_vector_store(chunks, embedding_model):

    return FAISS.from_documents(chunks, embedding_model)


def save_vector_store(vector_store, save_path: str = "faiss_index"):

    vector_store.save_local(save_path)


def load_vector_store(embedding_model, save_path: str = "faiss_index"):

    return FAISS.load_local(save_path, embedding_model, allow_dangerous_deserialization=True)


vector_store = build_vector_store(chunks, embedding_model)

print("Vector Store Created")
print("Total Vectors Indexed:", vector_store.index.ntotal)

save_vector_store(vector_store)  # optional: persists to ./faiss_index


Vector Store Created
Total Vectors Indexed: 10


## 5️. User Input Route — Query Embedding

Converts an incoming natural-language question into the same vector space as the
document chunks, so it can be compared against them.


In [38]:
def embed_query(embedding_model, query: str):

    return embedding_model.embed_query(query)


demo_query = "What is this document about?"
query_vector = embed_query(embedding_model, demo_query)
print(f"Query: {demo_query!r}")
print("Query vector dimension:", len(query_vector))


Query: 'What is this document about?'
Query vector dimension: 384


## 6️. Retrieval Module

Queries the vector store for the `top_k` chunks whose embeddings are closest to the
query embedding (lower FAISS L2 distance = more similar).


In [43]:
def retrieve_relevant_chunks(vector_store, query: str, top_k: int = 3):
    """Retrieve the top_k most relevant chunks for a query, with similarity scores."""
    return vector_store.similarity_search_with_score(query, k=top_k)


user_query = "Who is the main character?"
retrieved_results = retrieve_relevant_chunks(vector_store, user_query, top_k=5)

print(f"User Query: {user_query}\n")
for i, (chunk, score) in enumerate(retrieved_results):
    print(f"Result {i+1} (score: {score:.4f}):")
    print(chunk.page_content[:300])
    print("-" * 60)


User Query: Who is the main character?

Result 1 (score: 1.3579):
. It made me think even after I finished reading. Poirot is also a fun character. He is smart and a little funny at the same time. What I Did Not Like: My only problem with the book was the number of characters. There are twelve suspects and each one has a different name and background. I found it h
------------------------------------------------------------
Result 2 (score: 1.4040):
THEMES, NARRATIVES, AND INSIGHTS Justice vs. Law The main theme of this book is justice. The victim was a bad person who never faced punishment. Other people decided to punish him themselves. The book asks whether that was right or wrong. There is no easy answer. This is what makes the story so inte
------------------------------------------------------------
Result 3 (score: 1.4336):
. She created two very famous detective characters. One is Hercule Poirot and the other is Miss Marple. Both characters have appeared in many films and TV sho

## 7️. Grounded Generation

Combines the retrieved context chunks with the original query into a single prompt,
and passes it to a language model to produce an answer that is
grounded in the retrieved text rather than the model's parametric memory alone.


In [44]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

def get_llm_pipeline(model_name="Qwen/Qwen2.5-0.5B-Instruct"):
    hf_pipe = pipeline(
        task="text-generation",
        model=model_name,
        max_new_tokens=150,
        max_length=None,
        temperature=0.2,
        do_sample=True,
        return_full_text=False,
    )
    return HuggingFacePipeline(pipeline=hf_pipe)

def generate_answer(llm, query: str, retrieved_results):
    """Build a chat-structured prompt for an Instruct model to prevent prompt confusion."""
    context_text = "\n\n".join(chunk.page_content for chunk, score in retrieved_results)

    # Structure the inputs using Qwen's expected chat format tags
    prompt = (
        "<|im_start|>system\n"
        "You are a factual assistant. Read the provided context carefully and answer the user's question with a short, direct answer. "
        "Pay close attention to document metadata: distinguish between the author of a book being discussed and the person who submitted/wrote the document itself. "
        "Extract the exact answer from the context if possible. If the answer is not in the context, say you don't know.<|im_end|>\n"
        "<|im_start|>user\n"
        f"Context:\n{context_text}\n\n"
        f"Question: {query}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

    return llm.invoke(prompt).strip()


llm = get_llm_pipeline()
answer = generate_answer(llm, user_query, retrieved_results)
print(f"Query: {user_query}\n")
print(f"Generated Answer:\n{answer}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Query: Who is the main character?

Generated Answer:
The main character in this book is Hercule Poirot.


### Full pipeline in one call

Everything above wired together end-to-end — swap `my_query` for anything you like.


In [45]:
def rag_answer(query: str, vector_store, llm, top_k: int = 5):
    retrieved = retrieve_relevant_chunks(vector_store, query, top_k=top_k)
    return generate_answer(llm, query, retrieved), retrieved


my_query = "Summarize the key points in a few words."
answer, retrieved = rag_answer(my_query, vector_store, llm, top_k=3)

print("Q:", my_query)
print("A:", answer)


Q: Summarize the key points in a few words.
A: The book "Murder on the Orient Express" is a murder mystery written by Agatha Christie. It features a clever plot with a twist ending. The author uses simple language and straightforward writing style. The story is engaging and challenging to follow. The main character, Poirot, is described as intelligent and humorous. The book is recommended for those interested in suspense and thought-provoking stories.


In [49]:
import pandas as pd
import json
from datetime import datetime

# Initialize a simple log list in your session
if 'validation_logs' not in globals():
    validation_logs = []

def log_rag_turn(query: str, retrieved_chunks, generated_answer: str):
    """Logs the inputs and outputs of a RAG turn for evaluation."""
    log_entry = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "query": query,
        "retrieved_contexts": [
            {"score": float(score), "text": chunk.page_content[:200] + "..."}
            for chunk, score in retrieved_chunks
        ],
        "generated_answer": generated_answer
    }
    validation_logs.append(log_entry)

    # Save to a CSV file in your Colab files directory for easy download
    df = pd.DataFrame(validation_logs)
    df.to_csv("rag_validation_logs.csv", index=False)
    print(f" Logged turn successfully. Total logs: {len(validation_logs)}")

# --- Test Log Output ---
# Run your rag_answer function
answer, retrieved = rag_answer("Who is the main character in the book ", vector_store, llm, top_k=3)

# Log the results
log_rag_turn("Who is the main character in the book", retrieved, answer)

✅ Logged turn successfully. Total logs: 1


## 8️⃣ System Optimization Experiments

Three common levers for improving a RAG pipeline's retrieval quality:

1. **Chunk boundaries** — sweep chunk size / overlap and inspect the effect on chunk count and retrieval.
2. **Hybrid search** — combine keyword search (BM25) with dense vector search.
3. **Re-ranking** — add a cross-encoder re-ranker on top of the initial retrieval to reorder by finer-grained relevance.


### 8a. Chunk boundary sweep

In [46]:
chunk_configs = [
    {"chunk_size": 300, "chunk_overlap": 50},
    {"chunk_size": 500, "chunk_overlap": 100},
    {"chunk_size": 1000, "chunk_overlap": 150},
    {"chunk_size": 1500, "chunk_overlap": 200},
]

print(f"{'chunk_size':>10} {'overlap':>8} {'num_chunks':>11}")
for cfg in chunk_configs:
    trial_chunks = chunk_documents(documents, **cfg)
    print(f"{cfg['chunk_size']:>10} {cfg['chunk_overlap']:>8} {len(trial_chunks):>11}")

# Pick the configuration that best balances chunk count vs. context completeness
# for your corpus, then rebuild the vector store with build_vector_store(trial_chunks, embedding_model).


chunk_size  overlap  num_chunks
       300       50          32
       500      100          20
      1000      150          10
      1500      200           7


### 8b. Hybrid search — keyword (BM25) + vector

BM25 catches exact keyword/term matches that dense embeddings can sometimes miss
(rare proper nouns, codes, IDs), while vector search captures semantic similarity.
Combining both (reciprocal rank fusion) tends to be more robust than either alone.


In [47]:
from rank_bm25 import BM25Okapi
import numpy as np


class HybridRetriever:
    """Combines BM25 keyword search with FAISS dense vector search via reciprocal rank fusion."""

    def __init__(self, chunks, vector_store, k_rrf: int = 60):
        self.chunks = chunks
        self.vector_store = vector_store
        self.k_rrf = k_rrf
        tokenized_corpus = [c.page_content.lower().split() for c in chunks]
        self.bm25 = BM25Okapi(tokenized_corpus)

    def retrieve(self, query: str, top_k: int = 5, candidate_pool: int = 20):
        # Keyword ranking
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranking = np.argsort(bm25_scores)[::-1][:candidate_pool]

        # Vector ranking (map back to chunk indices via content match)
        vector_hits = self.vector_store.similarity_search_with_score(query, k=candidate_pool)
        content_to_idx = {c.page_content: i for i, c in enumerate(self.chunks)}
        vector_ranking = [content_to_idx[c.page_content] for c, _ in vector_hits
                           if c.page_content in content_to_idx]

        # Reciprocal Rank Fusion
        fused_scores = {}
        for rank, idx in enumerate(bm25_ranking):
            fused_scores[idx] = fused_scores.get(idx, 0) + 1.0 / (self.k_rrf + rank + 1)
        for rank, idx in enumerate(vector_ranking):
            fused_scores[idx] = fused_scores.get(idx, 0) + 1.0 / (self.k_rrf + rank + 1)

        ranked_indices = sorted(fused_scores, key=fused_scores.get, reverse=True)[:top_k]
        return [(self.chunks[i], fused_scores[i]) for i in ranked_indices]


hybrid_retriever = HybridRetriever(chunks, vector_store)
hybrid_results = hybrid_retriever.retrieve(user_query, top_k=5)

print(f"Hybrid results for: {user_query!r}\n")
for i, (chunk, score) in enumerate(hybrid_results):
    print(f"Result {i+1} (fused score: {score:.4f}):")
    print(chunk.page_content[:300])
    print("-" * 60)


Hybrid results for: 'Who is the main character?'

Result 1 (fused score: 0.0323):
THEMES, NARRATIVES, AND INSIGHTS Justice vs. Law The main theme of this book is justice. The victim was a bad person who never faced punishment. Other people decided to punish him themselves. The book asks whether that was right or wrong. There is no easy answer. This is what makes the story so inte
------------------------------------------------------------
Result 2 (fused score: 0.0318):
. Poirot finds out that Ratchett was actually a criminal. He had kidnapped and killed a child many years ago. He was never punished for it. This information changes the whole investigation. Turning Point and Main Message: At the end, Poirot gives a surprising answer. The ending is not like a normal 
------------------------------------------------------------
Result 3 (fused score: 0.0315):
. It made me think even after I finished reading. Poirot is also a fun character. He is smart and a little funny at the same time.

### 8c. Re-ranking with a cross-encoder

A cross-encoder scores the `(query, chunk)` pair jointly (rather than comparing
independent embeddings), which is slower but typically more accurate. Use it to
re-rank a small candidate pool pulled from the vector/hybrid retriever.


In [48]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank_chunks(query: str, candidates, top_k: int = 5):
    """Re-score (query, chunk) pairs with a cross-encoder and return the top_k reranked chunks."""
    pairs = [(query, chunk.page_content) for chunk, _ in candidates]
    scores = reranker.predict(pairs)

    reranked = sorted(zip([c for c, _ in candidates], scores), key=lambda x: x[1], reverse=True)
    return reranked[:top_k]


# Pull a wider candidate pool, then rerank down to the best few
candidate_pool = retrieve_relevant_chunks(vector_store, user_query, top_k=10)
reranked_results = rerank_chunks(user_query, candidate_pool, top_k=5)

print(f"Reranked results for: {user_query!r}\n")
for i, (chunk, score) in enumerate(reranked_results):
    print(f"Result {i+1} (rerank score: {score:.4f}):")
    print(chunk.page_content[:300])
    print("-" * 60)

# Feed the reranked context into the generator for a final, optimized answer
final_answer = generate_answer(llm, user_query, reranked_results)
print("\nFinal optimized answer:\n", final_answer)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Reranked results for: 'Who is the main character?'

Result 1 (rerank score: -5.5411):
. It made me think even after I finished reading. Poirot is also a fun character. He is smart and a little funny at the same time. What I Did Not Like: My only problem with the book was the number of characters. There are twelve suspects and each one has a different name and background. I found it h
------------------------------------------------------------
Result 2 (rerank score: -5.6405):
. She created two very famous detective characters. One is Hercule Poirot and the other is Miss Marple. Both characters have appeared in many films and TV shows. Her play called The Mousetrap has been running in London since 1952. That is a very long time. She passed away in 1976, but people still r
------------------------------------------------------------
Result 3 (rerank score: -6.0575):
THEMES, NARRATIVES, AND INSIGHTS Justice vs. Law The main theme of this book is justice. The victim was a bad person who n

## Summary

| Step | Component | Function |
|---|---|---|
| 1 | Ingestion | `ingest_documents()`, `load_document()`, `load_from_huggingface()` |
| 2 | Chunking | `chunk_documents()` |
| 3 | Embedding | `get_embedding_model()` |
| 4 | Vector store | `build_vector_store()`, `save_vector_store()`, `load_vector_store()` |
| 5 | Query route | `embed_query()` |
| 6 | Retrieval | `retrieve_relevant_chunks()` |
| 7 | Generation | `get_llm_pipeline()`, `generate_answer()`, `rag_answer()` |
| 8 | Optimization | chunk sweep, `HybridRetriever`, `rerank_chunks()` |

### RAG System Metrics Report

| Metric / Configuration Component | Value / Details |
| :--- | :--- |
| **Document Embedding Dimension** | 384 |
| **Total Text Chunks Created** | 10 |
| **Total Vectors Indexed** | 10 |
| **Vector Store Type** | FAISS (`faiss-cpu`) |
| **Target Chunk Size** | 1000 characters |
| **Chunk Overlap** | 150 characters |
| **Retrieval Top-K ($k$)** | 3 (Generation) / 5 (Retrieval Test) |
| **LLM Name & Pipeline** | `Qwen/Qwen2.5-0.5B-Instruct` |
| **Runtime / Indexing Summary** | Ingestion & Vector Indexing complete; local index persistence enabled (`./faiss_index`) |